# Land GHG inventory (vegetation) QC

Compares **every country** in a pipeline output parquet against the researcher
reference parquet, by `country × land_state_class`, summed over all years.

The pipeline's `land_state_class` (tree_loss / tree_gain / trees_remaining_trees /
non_trees_remaining_non_trees) is the collapse of the reference's `land_state`; we
derive the same collapse on the reference side via `classify()` so the two are
compared apples-to-apples. A `country × class × measure` is **flagged** only when it
differs by BOTH more than `REL_THRESHOLD` and more than `ABS_THRESHOLD` MgCO2e.

**Fill in the two paths in the config cell**, then run all.

In [ ]:
# === Fill these in ===
# researcher wide parquet (adm0-level, with veg__* columns + land_state classes)
REFERENCE_PARQUET = "/Users/justin.terry/Downloads/LULUCF__v1_0_0__for_figures__wide__20260617.parquet"
# our pipeline output (a GLOBAL run; a bbox run only covers the clipped countries)
OUTPUT_PARQUET = "../admin-land_ghg_inventory-vegetation-<version>.parquet"

# flag a country x land_state_class x measure only if BOTH thresholds are exceeded
REL_THRESHOLD = 0.02  # 2%
ABS_THRESHOLD = 1000.0  # MgCO2e

In [ ]:
import numpy as np
import pandas as pd

from pipelines.land_ghg_inventory.land_state_categories import (
    VEGETATION_CATEGORIES,
    classify,
)
from pipelines.land_ghg_inventory.stages import MEASURES

reference_raw = pd.read_parquet(REFERENCE_PARQUET)
ours_raw = pd.read_parquet(OUTPUT_PARQUET)
print("reference:", reference_raw.shape, "| ours:", ours_raw.shape)
print("measures:", MEASURES)

## Reference totals (country × land_state_class)

Map the researcher `veg__*` columns to our measure names, collapse `land_state` to
the 4 classes via the same `classify()` rule the pipeline uses, drop `excluded`,
and sum over years.

In [ ]:
# researcher wide-parquet column -> our canonical measure
REFERENCE_COLUMNS = {
    "gross_emissions_MgCO2e": "veg__gross_emissions__all_C_pools__all_gases__MgCO2e_yr",
    "gross_removals_MgCO2": "veg__gross_removals__all_C_pools__MgCO2_yr",
    "net_flux_MgCO2e": "veg__net_flux__all_C_pools__all_gases__MgCO2e_yr",
}

ref = reference_raw.rename(columns={src: name for name, src in REFERENCE_COLUMNS.items()})
ref["land_state_class"] = [
    VEGETATION_CATEGORIES[classify(detailed, broad)]
    for detailed, broad in zip(
        ref["land_state_detailed_class"], ref["land_state_broad_class"]
    )
]
ref = ref[ref["land_state_class"] != "excluded"]
reference_totals = (
    ref.groupby(["adm0", "land_state_class"])[MEASURES]
    .sum()
    .reset_index()
    .rename(columns={"adm0": "country"})
)
print("reference countries:", reference_totals["country"].nunique())
reference_totals.head()

## Our totals (country × land_state_class)

Country-level (adm0) rows only — `aoi_id` with no `.` (drops the adm1/adm2 rows).

In [ ]:
country_rows = ours_raw[~ours_raw["aoi_id"].str.contains(".", regex=False)]
our_totals = (
    country_rows.rename(columns={"aoi_id": "country"})
    .groupby(["country", "land_state_class"])[MEASURES]
    .sum()
    .reset_index()
)
print("our countries:", our_totals["country"].nunique())
our_totals.head()

## Compare all countries

Outer-merge so a `country × class` present on only one side is surfaced (treated as
flagged). For each measure, compute the symmetric relative difference and the
absolute difference; flag when both exceed the thresholds.

In [ ]:
keys = ["country", "land_state_class"]
merged = our_totals.merge(
    reference_totals, on=keys, how="outer", suffixes=("_ours", "_ref"), indicator=True
)
missing = merged["_merge"] != "both"


def symmetric_relative_difference(a, b):
    denom = (a.abs() + b.abs()) / 2
    return (a - b).abs() / denom.replace(0, np.nan)


parts = []
for measure in MEASURES:
    ours = merged[f"{measure}_ours"]
    ref_v = merged[f"{measure}_ref"]
    abs_diff = (ours - ref_v).abs()
    rel_diff = symmetric_relative_difference(ours, ref_v)
    flag = (rel_diff > REL_THRESHOLD) & (abs_diff > ABS_THRESHOLD)
    part = merged[keys].copy()
    part["measure"] = measure
    part["ours"] = ours
    part["ref"] = ref_v
    part["abs_diff"] = abs_diff
    part["rel_diff"] = rel_diff
    part["present"] = merged["_merge"].astype(str)
    part["flagged"] = flag | missing
    parts.append(part)

comparison = pd.concat(parts, ignore_index=True)
flagged = comparison[comparison["flagged"]].sort_values(
    "abs_diff", ascending=False, na_position="first"
)
print(
    f"flagged: {len(flagged)} of {len(comparison)} rows "
    f"(rel>{REL_THRESHOLD}, abs>{ABS_THRESHOLD} MgCO2e)"
)
flagged.head(40)

## Summary

In [ ]:
only_ours = merged.loc[merged["_merge"] == "left_only", keys]
only_ref = merged.loc[merged["_merge"] == "right_only", keys]
print(f"country x class only in OUR output: {len(only_ours)}")
print(f"country x class only in REFERENCE: {len(only_ref)}")

by_country = comparison.groupby("country")["flagged"].any()
matching = by_country[~by_country].index.tolist()
mismatched = by_country[by_country].index.tolist()
print(f"\ncountries fully matching: {len(matching)} / {len(by_country)}")
print(f"countries with >=1 flagged measure ({len(mismatched)}):")
print(mismatched)